# 02b — A global flood model on new floods
**"I just want to use a model."** In notebook 02a we trained a small regional model. Now we use the model
trained on the whole WorldFloods v2 dataset (~500 flood maps from all over the world; Portalés-Julià et al.,
2023, *Scientific Reports*) on floods it has never seen, far from Greece.

1. Download the trained model from HuggingFace and read its configuration.
2. Its two outputs: **cloud** and **water**.
3. One flood, two dates: the flood **peak** vs the **receding** water and its **flood traces**.
4. More floods the model has never seen.
5. From a raster to polygons (GeoJSON) that a GIS or an emergency service can use.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from georeader import plot
from course_utils import BANDS_S2, COLORS_PRED, PRED_CLASSES, ROOT, binary_metrics, find_event, read_event, save_fig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # the GPU if there is one

## 1. Download the trained model
The ml4floods models live in the HuggingFace repository
[isp-uv-es/ml4floods](https://huggingface.co/isp-uv-es/ml4floods). Each model is a folder with a
`config.json` and the weights `model.pt`: the same layout as the model we saved in notebook 02a.

In [ ]:
from huggingface_hub import hf_hub_download

EXPERIMENT = "WF2_unetv2_bgriswirs"
for filename in ("config.json", "model.pt"):
    hf_hub_download(repo_id="isp-uv-es/ml4floods", subfolder=f"models/{EXPERIMENT}", filename=filename,
                    local_dir=ROOT)   # -> ROOT/models/WF2_unetv2_bgriswirs/ (skipped if already there)

wf2_config = json.loads((ROOT / "models" / EXPERIMENT / "config.json").read_text())
wf2_config["model_params"]["hyperparameters"]

❓ Compare it with the configuration of 02a: same keys. Which architecture, bands and loss weights? It was
trained for up to 30 epochs on the whole WorldFloods v2 (`data_params`), and water errors weigh 6× more in its
loss (`pos_weight`): it favours **recall**, so we will use a threshold on P(water) higher than 0.5.

Its 6 bands (Blue, Green, Red, NIR, SWIR1, SWIR2) also exist in Landsat 8/9 → the same model works on both
satellites.

In [ ]:
from ml4floods.models.model_setup import get_channel_configuration_bands
from ml4floods.scripts.inference import load_inference_function

TH_WATER = 0.7
predict, config = load_inference_function(str(ROOT / "models" / EXPERIMENT), device_name=DEVICE,
                                          max_tile_size=1024,   # tiles of ≤1024 px to limit memory
                                          th_water=TH_WATER,
                                          distinguish_flood_traces=True)   # see section 3
channels = get_channel_configuration_bands(config.data_params.channel_configuration, collection_name="S2")
print("model input bands:", [BANDS_S2[c] for c in channels])

As in 02a: numpy → torch on the way in, `array_as_geotensor` puts the georeference back on the way out.

In [ ]:
from georeader.geotensor import GeoTensor


def wf2(s2: GeoTensor) -> tuple[GeoTensor, GeoTensor]:
    """Flood map {0: invalid, 1: land, 2: water, 3: cloud, 4: flood trace} and the two output probabilities
    (P(cloud), P(water); -1 where the input is invalid) of the global model on a 13-band Sentinel-2 image."""
    pred_class, prob = predict(torch.tensor(s2.values[channels]))
    return s2.array_as_geotensor(pred_class.numpy()), s2.array_as_geotensor(prob.numpy())

## 2. Two outputs: cloud and water
The WorldFloods v2 model is trained with **two outputs**, each with its own label: clear/cloud and land/water.
At inference, a rule combines **P(cloud)**, **P(water)** and the image brightness into the 3-class map
(land / water / cloud):

<img src="https://raw.githubusercontent.com/kipoju/geomundus-ml4floods/main/notebooks/figures/wf2_multioutput.png" width="900">

*Cloud-aware multioutput framework. Portalés-Julià et al. (2023), Scientific Reports.*

**Miandrivazo, Madagascar** (EMSR264, January 2018): a flash flood with sediment-laden water and scattered
clouds. The image is from the WorldFloods v2 **test** set (never used to
train or select the model).

The network gives **two numbers per pixel** (logits), each turned into a probability by its own sigmoid:
**P(cloud)** and **P(water)**. They are independent: a pixel can be water *and* cloud (thin cloud over a river).

In [ ]:
EVENT = "EMSR264_18MIANDRIVAZODETAIL_DEL_v2"
d = read_event(*find_event(EVENT), EVENT)
pred_class, prob = wf2(d["s2"])

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
plot.show((d["s2"].isel({"band": [3, 2, 1]}) / 3000).clip(0, 1), ax=axs[0], add_scalebar=True)
axs[0].set_title("Sentinel-2, 19 Jan 2018")
for ax, i, name, cmap in ((axs[1], 0, "P(cloud)", "Greys_r"), (axs[2], 1, "P(water)", "Blues")):
    plot.show(prob[i], ax=ax, mask=prob.values[i] < 0, cmap=cmap, vmin=0, vmax=1, add_colorbar_next_to=True)
    ax.set_title(f"output channel {i}: {name}")
for ax in axs:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

A simple rule turns the two probabilities into a map: **cloud** if P(cloud) > 0.5 and the pixel is bright;
else **water** if P(water) > `TH_WATER`; else **land**. With `distinguish_flood_traces=True`, water pixels
that do not look like water in the image (MNDWI < 0: wet soil, mud, sediment left behind) become **flood
traces**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
plot.plot_segmentation_mask(pred_class, COLORS_PRED, PRED_CLASSES, ax=ax)
ax.set_title(f"WF2 flood map, P(water) > {TH_WATER}")
plt.show()

## 3. One flood, two dates
The WorldFloods image is from **19 January**. But Copernicus EMS was activated on 5 January, and on
**9 January** Sentinel-2 had already seen the valley completely flooded (we downloaded that image for today,
on the same grid). The reference map was drawn by EMS analysts on a radar image of **12 January**.

❓ **Before running:** on which date will the model find more water? And which date will agree better with
the reference map?

In [ ]:
from georeader.rasterio_reader import RasterioReader

s2_peak = RasterioReader(str(ROOT / "data" / "extra" / "EMSR264_18MIANDRIVAZODETAIL_S2_2018-01-09.tif"))
s2_peak = s2_peak.isel({"band": list(range(len(BANDS_S2)))}).load()   # the 13 bands, as in WorldFloods
s2_peak = s2_peak.astype(np.float32)
dates = {"9 Jan 2018 (peak)": s2_peak, "19 Jan 2018 (WorldFloods)": d["s2"]}

gt = d["gt"]
ref, valid = gt[1] == 2, (gt[1] != 0) & (gt[0] != 2)   # reference: EMS map of 12 January
maps, rows = {}, {}
for name, s2 in dates.items():
    maps[name], _ = wf2(s2)
    flooded = (maps[name] == 2) | (maps[name] == 4)   # water + flood traces
    rows[name] = {**binary_metrics(flooded, ref, valid & (maps[name] != 3)),
                  "water (km²)": float((maps[name] == 2).sum()) * 1e-4,
                  "flood traces (km²)": float((maps[name] == 4).sum()) * 1e-4}
pd.DataFrame(rows).T.round(2)

In [ ]:
from course_utils import COLORS_CONFUSION, CONFUSION_CLASSES, confusion_classes

fig, axs = plt.subplots(2, 3, figsize=(20, 12))
for row, (name, s2) in zip(axs, dates.items()):
    plot.show((s2.isel({"band": [11, 7, 3]}) / 4500).clip(0, 1), ax=row[0])   # B11, B8, B4
    row[0].set_title(f"{name}: SWIR / NIR / Red")
    plot.plot_segmentation_mask(maps[name], COLORS_PRED, PRED_CLASSES, ax=row[1])
    row[1].set_title("WF2 flood map")
    flooded = (maps[name] == 2) | (maps[name] == 4)
    plot.plot_segmentation_mask(confusion_classes(flooded, ref, valid & (maps[name] != 3)), COLORS_CONFUSION,
                                CONFUSION_CLASSES, ax=row[2])
    row[2].set_title("vs EMS map of 12 Jan")
for ax in axs.ravel():
    ax.set_axis_off()
plt.tight_layout()
plt.show()

- **9 January**: the valley is under water and the model finds it. Most of its "false alarms" (red) are water
  that was no longer there (or that the radar did not see) when the reference was drawn three days later.
- **19 January**: the water is receding. The pink areas in SWIR / NIR / Red are mud and wet soil left behind.
  ❓ The "missed water" (green) was water on 12 January: is it a model error?
- **Flood traces** (orange): pixels the model calls water but that do not look like water in the image
  (MNDWI < 0), such as shallow muddy water and wet soil at the edges. A damage assessment wants them; a map of
  where the water is now does not.
- Same model, same place, ten days apart: **a reference map has a date**, and in a flood, *when* you look
  matters as much as *how*.

## 4. More floods the model has never seen
From the official WorldFloods v2 **test** set:
- **Ylitornio, Finland** (EMSR9284, May 2018): ice-jam flood of the Torne river in a boreal landscape.
- **Portopalo, Sicily** (EMSR333, Nov 2018): flash flood under a partly cloudy sky.

In [ ]:
NEW_FLOODS = {"Ylitornio (Finland)": "EMSR9284_01YLITORNIONORTHERN_DEL_MONIT01_v1",
              "Miandrivazo (Madagascar)": EVENT,
              "Portopalo (Sicily)": "EMSR333_02PORTOPALO_DEL_MONIT01_v1"}

results = {}
for name, event in NEW_FLOODS.items():
    dn = read_event(*find_event(event), event)
    pred, _ = wf2(dn["s2"])
    flooded = (pred == 2) | (pred == 4)
    m = binary_metrics(flooded, dn["gt"][1] == 2, (dn["gt"][1] != 0) & (dn["gt"][0] != 2))
    results[name] = dict(s2=dn["s2"], pred=pred, metrics=m)
    print(f"{name:26s} P {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}")

In [ ]:
# One column per flood: Sentinel-2 (SWIR / NIR / Red) on top, the model's flood map below
fig, axs = plt.subplots(2, len(results), figsize=(6.2 * len(results), 11.5))
for col, (name, r) in zip(axs.T, results.items()):
    plot.show((r["s2"].isel({"band": [11, 7, 3]}) / 4500).clip(0, 1), ax=col[0], add_scalebar=True)   # B11, B8, B4
    col[0].set_title(f"{name}\nSentinel-2 (B11, B8, B4)", fontsize=14)
    plot.plot_segmentation_mask(r["pred"], COLORS_PRED, PRED_CLASSES, ax=col[1])
    col[1].set_title(f"Flood map, P(water) > {TH_WATER}\nvs reference: IoU {r['metrics']['IoU']:.2f}", fontsize=14)
plt.tight_layout()
save_fig(fig, "dl_wf2_new_floods")

## 5. From raster to polygons
Emergency services work with vector layers (GIS, web maps). We turn the flooded pixels (water + flood traces)
into polygons with georeader and save them as GeoJSON: open the file in QGIS or on geojson.io.

In [ ]:
import geopandas as gpd
from georeader.save import save_cog
from georeader.vectorize import get_polygons

r = results["Miandrivazo (Madagascar)"]
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
flooded = ((r["pred"] == 2) | (r["pred"] == 4)).astype(np.uint8)   # GeoTensor: pixels → map coordinates come for free
save_cog(flooded, str(out_dir / f"{EVENT}_{EXPERIMENT}_water.tif"), descriptions=["flooded"])   # raster
polygons = get_polygons(flooded, min_area=25)   # in pixels: drop polygons smaller than 25 px (0.25 ha)
flood = gpd.GeoDataFrame(geometry=polygons, crs=r["s2"].crs)
print(f"{len(flood)} polygons, total area {flood.area.sum() / 1e6:.1f} km²")

out_file = out_dir / f"{EVENT}_{EXPERIMENT}.geojson"
flood.to_crs("EPSG:4326").to_file(out_file, driver="GeoJSON")
print("saved", out_file)

fig, ax = plt.subplots(figsize=(10, 9))
plot.show((r["s2"].isel({"band": [3, 2, 1]}) / 3000).clip(0, 1), ax=ax, add_scalebar=True)
flood.boundary.plot(ax=ax, color="cyan", linewidth=0.6)
ax.set_title(f"Miandrivazo: {len(flood)} flood polygons ({flood.area.sum() / 1e6:.1f} km²) → GeoJSON")
save_fig(fig, "dl_polygons")

**Take-aways**
- A model trained on many floods worldwide works on floods far from anything we trained on today: weights and
  configuration from HuggingFace → Sentinel-2 (or Landsat) image → flood map → polygons for a GIS.
- It gives two outputs, cloud and water: dense clouds still hide the water (optical sensor) → SAR or the next
  clear image.
- A flood changes in days: pick the image date with care, and remember that reference maps have a date too.